# Taller IEEE: Clasificación de la Madurez del Banano mediante Adaptación de Dominio CREDA, Cuantización con ExecuTorch e Inferencia Interactiva con Gradio
## Del entrenamiento robusto a la iluminación al despliegue en Hugging Face Spaces

**Profesores responsables:** Lucas Iturriago — Luis Chuqimarca
**Evento:** Taller IEEE

### Resumen
En el presente taller se desarrolla el ciclo completo de un sistema de visión artificial para la clasificación de la madurez del banano en cuatro estados: **Green, Partially Green, Ripe y Overripe**. El flujo comprende la adaptación de dominio entre condiciones de iluminación, la evaluación del rendimiento, la exportación del modelo cuantizado a ExecuTorch y su despliegue interactivo en Hugging Face Spaces mediante Gradio.

### Objetivos de aprendizaje
1. Comprender los fundamentos de la adaptación de dominio no supervisada con el algoritmo CREDA.
2. Evaluar el rendimiento de un clasificador bajo variación de iluminación mediante métricas y visualizaciones estandarizadas.
3. Exportar el modelo a formato ExecuTorch con cuantización estática INT8 (XNNPACK).
4. Desplegar un servicio de inferencia interactiva en Hugging Face Spaces con Gradio.

### Estructura del cuaderno
- **Sección 1. Entrenamiento** del modelo con adaptación de dominio CREDA.
- **Sección 2. Evaluación del rendimiento** en el dominio objetivo.
- **Sección 3. Exportación a ExecuTorch cuantizado** (FP32 a INT8).
- **Sección 4. Inferencia interactiva** en Hugging Face Spaces con Gradio.

### Requisitos
Entorno Kaggle, Google Colab o local, con GPU recomendada. El conjunto de datos se obtiene de forma portable mediante `kagglehub` (véase la etapa de preparación); no se asume ninguna ruta fija de Kaggle. El único requisito previo es disponer de credenciales de Kaggle para la descarga del conjunto `lucasiturriago/bananaripeness`.

### Nota metodológica
Este cuaderno consolida en un único documento los flujos anteriormente distribuidos en los cuadernos `train`, `performance`, `export` e `interactive` (conservados en formato `.py` como referencia). La aplicación de demostración ha sido corregida para operar exclusivamente sobre las cuatro clases objeto de estudio.


## Preparación del entorno
En esta etapa se verifica la instalación del paquete `banana_creda` y la disponibilidad de GPU, se definen las constantes globales del taller y se adquiere el conjunto de datos de forma portable entre Kaggle, Colab y entorno local. Las rutas del conjunto de datos se resuelven en tiempo de ejecución y se aplican a las configuraciones de cada sección.


In [ ]:
# Verificación del entorno e instalación del proyecto
# En Kaggle o Google Colab, ejecútese una única vez:
# !git clone https://github.com/UN-GCPDS/BananaRipenessCREDA.git
# %cd BananaRipenessCREDA
# !pip install -e . -q

import os
import torch

print("Directorio de trabajo:", os.getcwd())
print("Módulo del proyecto disponible:", os.path.isdir("src/banana_creda"))
print("Versión de PyTorch:", torch.__version__)
print("Aceleración CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Dispositivo:", torch.cuda.get_device_name(0))


In [ ]:
# Constantes globales del taller
from pathlib import Path

CLASS_NAMES = ["Green", "Partially Green", "Ripe", "Overripe"]
CONFIG_TRAIN = "configs/training/mobilenetv3/experiment_3.yaml"
CONFIG_EVAL = "configs/evaluation/mobilenetv3/evaluation_medium.yaml"
CKPT_REFERENCE = "/kaggle/input/models/lucasiturriago/bananaripeness/pytorch/mobilenetv3/1/ckpt_exp_3.pth"
DATASET_SLUG = "lucasiturriago/bananaripeness"

assert Path(CONFIG_TRAIN).exists(), f"No se encontró {CONFIG_TRAIN}"
assert Path(CONFIG_EVAL).exists(), f"No se encontró {CONFIG_EVAL}"
print("Clases objeto de estudio:", CLASS_NAMES)
print("Configuración de entrenamiento:", CONFIG_TRAIN)
print("Configuración de evaluación:", CONFIG_EVAL)
print("Identificador del conjunto de datos:", DATASET_SLUG)


In [ ]:
# Adquisición portable del conjunto de datos mediante kagglehub
# En Kaggle, el conjunto puede encontrarse ya montado en /kaggle/input; en Colab o en
# entorno local se descarga automáticamente. En ningún caso se asume una ruta fija.
import os
from pathlib import Path

# !pip install -q kagglehub
import kagglehub

KAGGLE_INPUT_CANDIDATES = [
    Path("/kaggle/input/datasets/lucasiturriago/bananaripeness"),
    Path("/kaggle/input/bananaripeness"),
]

def _contains_domains(root: Path) -> bool:
    return (root / "Original").is_dir() and (root / "Medium_Variation").is_dir()

def resolve_dataset_root():
    for candidate in KAGGLE_INPUT_CANDIDATES:
        if _contains_domains(candidate):
            return candidate
    downloaded = Path(kagglehub.dataset_download(DATASET_SLUG))
    print("Conjunto descargado en:", downloaded)
    if _contains_domains(downloaded):
        return downloaded
    # Búsqueda en profundidad ante estructuras anidadas (p. ej. versiones del dataset).
    for sub in [downloaded] + [p for p in downloaded.iterdir() if p.is_dir()]:
        if _contains_domains(sub):
            return sub
        for subsub in [p for p in sub.iterdir() if p.is_dir()] if sub.is_dir() else []:
            if _contains_domains(subsub):
                return subsub
    raise FileNotFoundError(f"No se localizaron las carpetas 'Original' y 'Medium_Variation' bajo {downloaded}.")

DATASET_ROOT = resolve_dataset_root()
SOURCE_DIR = DATASET_ROOT / "Original"
TARGET_DIR = DATASET_ROOT / "Medium_Variation"

os.environ["BANANA_DATASET_ROOT"] = str(DATASET_ROOT)
os.environ["BANANA_SOURCE_DIR"] = str(SOURCE_DIR)
os.environ["BANANA_TARGET_DIR"] = str(TARGET_DIR)

print("Path to dataset files:", DATASET_ROOT)
print("Dominio fuente:", SOURCE_DIR)
print("Dominio objetivo:", TARGET_DIR)

# Genera una configuración de sesión con las rutas resueltas, sin modificar los YAML originales.
import tempfile, yaml

def get_session_config(base_yaml: str) -> str:
    with open(base_yaml) as f:
        config_dict = yaml.safe_load(f)
    config_dict["data"]["source_data_dir"] = str(SOURCE_DIR)
    config_dict["data"]["target_data_dir"] = str(TARGET_DIR)
    tmp = tempfile.NamedTemporaryFile(mode="w", suffix=".yaml", delete=False, prefix="session_")
    yaml.safe_dump(config_dict, tmp)
    tmp.close()
    return tmp.name

SESSION_TRAIN = get_session_config(CONFIG_TRAIN)
SESSION_EVAL = get_session_config(CONFIG_EVAL)
print("Configuración de sesión (entrenamiento):", SESSION_TRAIN)
print("Configuración de sesión (evaluación):", SESSION_EVAL)


# 1. Entrenamiento del modelo con adaptación de dominio CREDA
## 1.1. Fundamento
El modelo se entrena sobre el dominio fuente (iluminación estándar, con etiquetas) y se adapta al dominio objetivo (iluminación variable, sin etiquetas) mediante la pérdida Class-Regularized Entropy Domain Adaptation (CREDA), complementada con minimización de entropía y ponderación por incertidumbre. La arquitectura empleada es MobileNetV3, seleccionada por su eficiencia para despliegue en dispositivos con recursos limitados.

## 1.2. Procedimiento
- **Paso 1.1.** Revisión de la configuración del experimento.
- **Paso 1.2.** Carga de los dominios fuente y objetivo e inspección visual (rutas resueltas en la preparación).
- **Paso 1.3.** Instanciación del modelo y ejecución del entrenamiento.


In [ ]:
# 1.1. Revisión de la configuración del experimento
# La configuración se valida mediante esquemas Pydantic (véase src/banana_creda/config.py).
from pathlib import Path

print("===== Configuración de entrenamiento =====")
print(Path(CONFIG_TRAIN).read_text())


In [ ]:
# 1.2. Carga de los dominios fuente y objetivo e inspección visual
from banana_creda.config import ExperimentConfig
from banana_creda.data.loader import BananaDataLoader
import matplotlib.pyplot as plt
import numpy as np

# Se emplea la configuración de sesión, con las rutas resueltas por kagglehub.
cfg = ExperimentConfig.from_yaml(SESSION_TRAIN)
data_manager = BananaDataLoader(cfg.data)

src_train, src_val, src_test, class_names = data_manager.get_split_loaders(cfg.data.source_data_dir)
tgt_train, tgt_val, tgt_test, _ = data_manager.get_split_loaders(cfg.data.target_data_dir, cfg.data.use_lime_on_target)

print("Clases identificadas:", class_names)
print(f"Número de lotes — fuente (entrenamiento): {len(src_train)} | objetivo (entrenamiento): {len(tgt_train)}")

batch = next(iter(tgt_test))
images, labels = batch[0][:8], batch[1][:8]
mean = np.array(cfg.data.imagenet_mean)
std = np.array(cfg.data.imagenet_std)
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, img, lab in zip(axes.flat, images, labels):
    restored = img.permute(1, 2, 0).numpy() * std + mean
    ax.imshow(np.clip(restored, 0, 1))
    ax.set_title(class_names[int(lab)])
    ax.axis("off")
fig.suptitle("Dominio objetivo (conjunto de prueba, variación media de iluminación)")
fig.tight_layout()
plt.show()


In [ ]:
# 1.3. Instanciación del modelo y ejecución del entrenamiento
from banana_creda.models.backbones import BananaModel
from scripts.da_train import run_da_experiment
import glob

model = BananaModel(cfg.model)
total_params = sum(p.numel() for p in model.parameters())
print(f"Arquitectura: {cfg.model.backbone} | Clases: {cfg.model.num_classes} | Parámetros: {total_params / 1e6:.2f} M")

# Ejecución del experimento CREDA con la configuración de sesión (rutas portables).
# En caso de disponer de tiempo limitado, puede omitirse esta celda y emplearse el punto de control de referencia en la Sección 2.
run_da_experiment(SESSION_TRAIN)

print("\nArchivos generados:")
for path in sorted(glob.glob("outputs/mobilenetv3/experiment_3/*")):
    print(" -", path)


# 2. Evaluación del rendimiento en el dominio objetivo
## 2.1. Fundamento
La evaluación se realiza sobre el conjunto de prueba del dominio objetivo (variación media de iluminación). Se reportan la exactitud global, la precisión, la exhaustividad (recall) y la medida F1, tanto en promedio macro como por clase. El análisis se complementa con la matriz de confusión, las curvas ROC por clase y la visualización del espacio latente mediante UMAP, que evidencia el grado de alineación entre dominios.

## 2.2. Procedimiento
- **Paso 2.1.** Ejecución de la rutina oficial de evaluación.
- **Paso 2.2.** Análisis del reporte por clase y de las visualizaciones generadas.


In [ ]:
# 2.1. Ejecución de la rutina oficial de evaluación
import os
from scripts.evaluation import run_evaluation

CKPT = CKPT_REFERENCE if os.path.exists(CKPT_REFERENCE) else "outputs/mobilenetv3/experiment_3/model_final.pth"
print("Punto de control evaluado:", CKPT)

# Se evalúa con la configuración de sesión, cuyas rutas fueron resueltas por kagglehub.
run_evaluation(SESSION_EVAL, CKPT, "outputs/mobilenetv3/da/medium")


In [ ]:
# 2.2. Análisis del reporte por clase y de las visualizaciones generadas
import glob
import torch
from PIL import Image
from banana_creda.config import ExperimentConfig
from banana_creda.data.loader import BananaDataLoader
from banana_creda.models.backbones import BananaModel
from banana_creda.utils.metrics import MetricTracker
from banana_creda.utils.visualizer import BananaVisualizer

cfg_eval = ExperimentConfig.from_yaml(SESSION_EVAL)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loader = BananaDataLoader(cfg_eval.data)
_, _, src_test, class_names = loader.get_split_loaders(cfg_eval.data.source_data_dir)
_, _, tgt_test, _ = loader.get_split_loaders(cfg_eval.data.target_data_dir, cfg_eval.data.use_lime_on_target)

model_eval = BananaModel(cfg_eval.model).to(device)
model_eval.load_state_dict(torch.load(CKPT, map_location=device))
model_eval.eval()

visualizer = BananaVisualizer(device=device, output_dir="outputs/mobilenetv3/da/medium/evaluation")
y_true, y_pred, _, _, _ = visualizer._get_inference_data(model_eval, tgt_test)
metrics = MetricTracker.compute_full_metrics(torch.from_numpy(y_pred), torch.from_numpy(y_true), len(class_names), device)
MetricTracker.print_full_report("Conjunto de prueba del dominio objetivo (variación media)", metrics, class_names)

for figure in sorted(glob.glob("outputs/mobilenetv3/da/medium/evaluation/*.png")):
    print(figure)
    display(Image.open(figure))


# 3. Exportación del modelo a ExecuTorch con cuantización INT8
## 3.1. Fundamento
Con el fin de habilitar el despliegue en dispositivos con recursos restringidos, el modelo en precisión flotante de 32 bits (FP32) se exporta mediante `torch.export`, se calibra con una muestra representativa de 100 imágenes y se cuantiza a enteros de 8 bits (INT8) con el backend XNNPACK, siguiendo el flujo PT2E (Prepare-Convert). El artefacto resultante, en formato `.pte`, conserva la interfaz tensorial original y reduce sustancialmente el tamaño y la latencia de inferencia.

## 3.2. Procedimiento
- **Paso 3.1.** Exportación, calibración, cuantización y compilación del modelo.
- **Paso 3.2.** Comparación cuantitativa entre los modelos FP32 e INT8.


In [ ]:
# 3.1. Exportación, calibración, cuantización y compilación del modelo
# Requiere los componentes opcionales de conversión (ejecútese una única vez):
# !pip install -e .[convert] -q

import os
from pathlib import Path
import torch
from banana_creda.config import ExperimentConfig
from banana_creda.models.backbones import BananaModel
from banana_creda.utils.metrics import MetricTracker
from banana_creda.utils.reproducibility import set_seed
from scripts.convert_and_quantize import ExecuTorchWrapper, get_dataloaders, evaluate_pytorch_model
from executorch.backends.xnnpack.quantizer.xnnpack_quantizer import XNNPACKQuantizer, get_symmetric_quantization_config
from torchao.quantization.pt2e.quantize_pt2e import prepare_pt2e, convert_pt2e
from executorch.exir import to_edge_transform_and_lower
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner

cfg_conv = ExperimentConfig.from_yaml(SESSION_EVAL)
set_seed(cfg_conv.training.seed)
device_cpu = torch.device("cpu")  # La cuantización y la compilación se ejecutan en CPU.

train_loader, _, test_loader, class_names = get_dataloaders(cfg_conv)
model_conv = BananaModel(cfg_conv.model)
model_conv.load_state_dict(torch.load(CKPT, map_location=device_cpu))
wrapped = ExecuTorchWrapper(model_conv).eval()

fp32_pred, fp32_true, fp32_latency = evaluate_pytorch_model(wrapped, test_loader, device_cpu)
fp32_metrics = MetricTracker.compute_full_metrics(fp32_pred, fp32_true, len(class_names), device_cpu)
MetricTracker.print_full_report("Modelo base (FP32)", fp32_metrics, class_names)

example_inputs = (torch.randn(1, 3, 224, 224),)
exported = torch.export.export(wrapped, example_inputs)
quantizer = XNNPACKQuantizer()
quantizer.set_global(get_symmetric_quantization_config(is_per_channel=True, is_dynamic=False))
prepared = prepare_pt2e(exported.module(), quantizer)

with torch.no_grad():
    calibrated = 0
    for batch in train_loader:
        for i in range(batch[0].size(0)):
            prepared(batch[0][i:i + 1])
            calibrated += 1
            if calibrated >= 100:
                break
        if calibrated >= 100:
            break
print(f"Calibración completada con {calibrated} imágenes.")

quantized = convert_pt2e(prepared)
quantized_exported = torch.export.export(quantized, example_inputs)
executorch_program = to_edge_transform_and_lower(quantized_exported, partitioner=[XnnpackPartitioner()]).to_executorch()

output_dir = Path(cfg_conv.experiment.output_dir)
output_dir.mkdir(parents=True, exist_ok=True)
PTE_PATH = output_dir / "model_quantized_xnnpack.pte"
with open(PTE_PATH, "wb") as f:
    f.write(executorch_program.buffer)
print(f"Modelo cuantizado almacenado en: {PTE_PATH} ({os.path.getsize(PTE_PATH) / 1024 / 1024:.2f} MB)")


In [ ]:
# 3.2. Comparación cuantitativa entre los modelos FP32 e INT8
from banana_creda.utils.metrics import MetricTracker
from scripts.convert_and_quantize import evaluate_executorch_model, print_comparison_table

int8_pred, int8_true, int8_latency = evaluate_executorch_model(PTE_PATH, test_loader)
int8_metrics = MetricTracker.compute_full_metrics(int8_pred, int8_true, len(class_names), device_cpu)
MetricTracker.print_full_report("Modelo cuantizado ExecuTorch (INT8, XNNPACK)", int8_metrics, class_names)

print_comparison_table(fp32_metrics, int8_metrics, class_names)
print(f"Latencia — FP32: {fp32_latency:.2f} ms/muestra | INT8: {int8_latency:.2f} ms/muestra")


# 4. Inferencia interactiva en Hugging Face Spaces con Gradio
## 4.1. Fundamento
El modelo cuantificado se expone como un servicio interactivo mediante Gradio, el cual permite cargar una fotografía y obtener la distribución de probabilidad sobre las cuatro clases. El despliegue se realiza en Hugging Face Spaces con el SDK de Docker, a fin de incluir el tiempo de ejecución de ExecuTorch. La aplicación prioriza el artefacto `.pte` y emplea los pesos `.pth` como mecanismo de respaldo.

## 4.2. Procedimiento
- **Paso 4.1.** Validación local de la interfaz Gradio.
- **Paso 4.2.** Generación de los archivos del Space (`app.py`, `Dockerfile`, `README.md`) y publicación.


In [ ]:
# 4.1. Validación local de la interfaz Gradio
# !pip install -q gradio pillow

import numpy as np
import torch
from PIL import Image
from banana_creda.config import ExperimentConfig
from banana_creda.data.loader import BananaDataLoader
from banana_creda.models.backbones import BananaModel

cfg_inf = ExperimentConfig.from_yaml(SESSION_EVAL)
device_inf = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_inf = BananaModel(cfg_inf.model).to(device_inf)
model_inf.load_state_dict(torch.load(CKPT, map_location=device_inf))
model_inf.eval()

try:
    _, _, _, CLASS_NAMES = BananaDataLoader(cfg_inf.data).get_split_loaders(cfg_inf.data.source_data_dir)
    CLASS_NAMES = list(CLASS_NAMES)
except Exception:
    pass

MEAN = np.array(cfg_inf.data.imagenet_mean, dtype=np.float32)
STD = np.array(cfg_inf.data.imagenet_std, dtype=np.float32)

def predict_maturity(image: Image.Image):
    if image is None:
        return {}
    resized = image.convert("RGB").resize((cfg_inf.data.img_size, cfg_inf.data.img_size))
    array = np.array(resized, dtype=np.float32) / 255.0
    array = (array - MEAN) / STD
    tensor = torch.from_numpy(np.transpose(array, (2, 0, 1))).unsqueeze(0).to(device_inf)
    with torch.no_grad():
        probabilities = torch.softmax(model_inf(tensor)[0], dim=0).cpu().numpy()
    return {CLASS_NAMES[i]: float(probabilities[i]) for i in range(len(CLASS_NAMES))}

import gradio as gr
demo = gr.Interface(
    fn=predict_maturity,
    inputs=gr.Image(type="pil", label="Fotografía del banano"),
    outputs=gr.Label(num_top_classes=4, label="Probabilidades por estado de madurez"),
    title="Madurez del banano — CREDA",
    description="Clasificación en Green, Partially Green, Ripe y Overripe. Modelo MobileNetV3 adaptado a variaciones de iluminación.",
)
demo.launch(share=True)


In [ ]:
# 4.2. Generación de los archivos del Space y publicación
import os
import shutil

HF_USER = "usuario_hf"  # Sustitúyase por el nombre de usuario de Hugging Face.
SPACE_NAME = "banana-creda-madurez"  # Nombre del Space (debe crearse previamente como Space Docker vacío).
os.makedirs(SPACE_NAME, exist_ok=True)

app_content = '''import os
import numpy as np
import torch
from PIL import Image
import gradio as gr

try:
    from executorch.runtime import Runtime
    EXECUTORCH_AVAILABLE = True
except ImportError:
    EXECUTORCH_AVAILABLE = False

PTE_PATH = "banana_quantized.pte"
PTH_PATH = "model_final.pth"
LABELS = ["Green", "Partially Green", "Ripe", "Overripe"]
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)
IMAGE_SIZE = 224

_torch_model = None
_et_method = None

def load_torch_fallback():
    global _torch_model
    if _torch_model is not None:
        return _torch_model
    if not os.path.exists(PTH_PATH):
        return None
    from banana_creda.config import ExperimentConfig
    from banana_creda.models.backbones import BananaModel
    config = ExperimentConfig.from_yaml("evaluation_medium.yaml")
    model = BananaModel(config.model)
    model.load_state_dict(torch.load(PTH_PATH, map_location="cpu"))
    _torch_model = model.eval()
    return _torch_model

def load_executorch():
    global _et_method
    if _et_method is not None:
        return _et_method
    if not (EXECUTORCH_AVAILABLE and os.path.exists(PTE_PATH)):
        return None
    runtime = Runtime.get()
    program = runtime.load_program(PTE_PATH)
    _et_method = program.load_method("forward")
    return _et_method

def preprocess(image):
    image = image.convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE))
    array = np.array(image, dtype=np.float32) / 255.0
    array = (array - MEAN) / STD
    return torch.from_numpy(np.transpose(array, (2, 0, 1))).unsqueeze(0).contiguous()

def predict(image):
    if image is None:
        return {}
    tensor = preprocess(image)
    method = load_executorch()
    if method is not None:
        outputs = method.execute([tensor])
        logits = outputs[0] if isinstance(outputs, (list, tuple)) else outputs
        if isinstance(logits, np.ndarray):
            logits = torch.from_numpy(logits)
        probabilities = torch.softmax(logits[0], dim=0).detach().numpy()
        return {LABELS[i]: float(probabilities[i]) for i in range(len(LABELS))}
    model = load_torch_fallback()
    if model is None:
        return {"Modelo no disponible": 1.0}
    with torch.no_grad():
        probabilities = torch.softmax(model(tensor)[0], dim=0).numpy()
    return {LABELS[i]: float(probabilities[i]) for i in range(len(LABELS))}

with gr.Blocks(title="Madurez del banano — CREDA + ExecuTorch") as demo:
    gr.Markdown("# Clasificación de la madurez del banano")
    gr.Markdown("Modelo MobileNetV3 con adaptación de dominio CREDA, cuantizado a INT8 con ExecuTorch (XNNPACK). Clases: **Green, Partially Green, Ripe, Overripe**.")
    gr.Markdown("Taller IEEE — Profesores Lucas Iturriago y Luis Chuqimarca.")
    with gr.Row():
        image_input = gr.Image(type="pil", label="Fotografía del banano")
        label_output = gr.Label(num_top_classes=4, label="Probabilidades por estado de madurez")
    classify_button = gr.Button("Clasificar")
    classify_button.click(predict, inputs=image_input, outputs=label_output)

if __name__ == "__main__":
    demo.launch(server_name="0.0.0.0", server_port=7860, show_error=True)
'''

dockerfile_content = '''FROM python:3.10-slim
ENV PYTHONDONTWRITEBYTECODE=1 PYTHONUNBUFFERED=1
RUN apt-get update && apt-get install -y --no-install-recommends build-essential libgl1 libglib2.0-0 git && rm -rf /var/lib/apt/lists/*
WORKDIR /app
RUN useradd -m -u 1000 user
USER user
ENV HOME=/home/user PATH=/home/user/.local/bin:$PATH
WORKDIR $HOME/app
RUN pip install --no-cache-dir --upgrade pip && pip install --no-cache-dir -q torch torchvision numpy pillow gradio executorch torchao
COPY --chown=user app.py ./app.py
COPY --chown=user *.pte ./
COPY --chown=user *.pth ./
COPY --chown=user *.yaml ./
EXPOSE 7860
CMD ["python", "app.py"]
'''

readme_content = f'''---
title: {SPACE_NAME}
emoji: "\U0001F34C"
colorFrom: yellow
colorTo: green
sdk: docker
app_file: app.py
pinned: false
---

# Clasificación de la madurez del banano — CREDA + ExecuTorch INT8
Servicio interactivo de clasificación en cuatro estados: **Green, Partially Green, Ripe, Overripe**.
Taller IEEE — Profesores Lucas Iturriago y Luis Chuqimarca.
'''

with open(os.path.join(SPACE_NAME, "app.py"), "w", encoding="utf-8") as f:
    f.write(app_content)
with open(os.path.join(SPACE_NAME, "Dockerfile"), "w", encoding="utf-8") as f:
    f.write(dockerfile_content)
with open(os.path.join(SPACE_NAME, "README.md"), "w", encoding="utf-8") as f:
    f.write(readme_content)

for artifact, target in [(str(PTE_PATH), "banana_quantized.pte"), (CKPT, "model_final.pth"), (CONFIG_EVAL, "evaluation_medium.yaml")]:
    if artifact and os.path.exists(artifact):
        shutil.copy(artifact, os.path.join(SPACE_NAME, target))
        print(f"Artefacto incorporado: {target}")
    else:
        print(f"Advertencia: no se encontró el artefacto {artifact}.")

print("\nContenido del directorio del Space:", sorted(os.listdir(SPACE_NAME)))
print(f"Dirección del Space: https://huggingface.co/spaces/{HF_USER}/{SPACE_NAME}")


In [ ]:
# 4.3. Publicación del Space (requiere un token de Hugging Face con permiso de escritura)
from huggingface_hub import notebook_login

try:
    notebook_login()
except Exception:
    print("Autenticación omitida: el procedimiento continuará solicitando el token durante la publicación.")

# Créese previamente un Space Docker vacío en https://huggingface.co/spaces y ejecútese a continuación:
# !git clone https://huggingface.co/spaces/usuario_hf/banana-creda-madurez space_clone
# Trasladar el contenido generado al clon, o publicar directamente desde el directorio del Space:
# %cd banana-creda-madurez
# !git lfs track "*.pte" "*.pth"
# !git add .gitattributes app.py Dockerfile README.md *.pte *.pth *.yaml
# !git commit -m "Despliegue del clasificador de madurez del banano (CREDA, INT8)"
# !git branch -M main
# !git push -u origin main
# %cd ..


## Conclusiones
En el desarrollo del taller se ha completado el ciclo de un sistema de clasificación de la madurez del banano: (1) entrenamiento con adaptación de dominio CREDA sobre MobileNetV3; (2) evaluación del rendimiento en condiciones de iluminación variable; (3) exportación del modelo cuantizado a ExecuTorch en formato INT8; y (4) despliegue de la inferencia interactiva en Hugging Face Spaces con Gradio.

### Actividades propuestas
1. Determinar cuál de las cuatro clases presenta la mayor variación de la medida F1 tras la cuantización y analizar sus causas.
2. Repetir las Secciones 2 y 3 con las arquitecturas ResNet o EfficientNet y comparar el equilibrio entre exactitud, tamaño y latencia.
3. Evaluar el servicio desplegado con imágenes propias bajo distintas condiciones de iluminación y valorar el efecto de la adaptación de dominio.
